# Round trip: every assembled data source for one tile / year

Reconstructs a raster for **every column** of the assembled panel for a
single `(ix, iy)` tile and a single `year`, straight from `pixel_id`, and lays
them all out in one figure. This is the round-trip check that the flat panel can
be folded back to the pixel grid it came from.

The panel is written by the DuckDB SQL assembly engine
(`src/data/assemble/sql_engine.py`) onto the canonical EPSG:6933 EASE grid
(`src/data/common/geobox/canonical.py`). Tile `(ix, iy)` indexing is always
native-resolution (2048x2048 native 1 km cells, `src/data/assemble/tiles.py`),
regardless of `grid_label` -- coarser grids just reproject that same native
tile geobox down with `GeoBox.zoom_to()` before packing `pixel_id`
(`[ix:16 | iy:16 | local:32]`), matching `sql_engine._pixel_id_sql`'s ragged
per-tile block aggregation. This notebook mirrors that with
`tile_native.zoom_to(resolution=...)` + `make_pixel_ids()`, so it works for
any `grid_label`, not just the native `"1km"`.

Set `DATA` below to wherever you synced `assembled/` to (HPC path or a local
mirror) and pick a `grid_label` you actually have on disk.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pyproj
from odc.geo.xr import ODCExtensionDa  # noqa: F401  (registers the .odc accessor)

from src.data.common.geobox.canonical import canonical_ease_geobox
from src.data.assemble.tiles import create_tile_geobox
from src.data.assemble.utils import make_pixel_ids, geobox_spatial_dims
from src.data.assemble.constants import GRID_RESOLUTIONS_M, DEFAULT_TILE_SIZE

In [ ]:
# --- parameters --------------------------------------------------------------
PROJECT = "/Users/felixschulz/Library/CloudStorage/OneDrive-Personal/Dokumente/Job/UNI/Basel/Research/growth-and-temperature"
DATA = f"{PROJECT}/data"  # local mirror of HPC's data_nobackup/ (see orchestration/configs/data.local.yaml)

ix, iy = 0, 8          # tile index into the native-resolution EASE tile grid -- contains Switzerland
year = 2013            # panel year to plot (ignored for time-invariant columns)
grid_label = "10km"
shake_label = "base"

# COPY ... PARTITION_BY (ix, iy) FILENAME_PATTERN 'data_{i}' writes one or more
# data_0.parquet, data_1.parquet, ... under multiple writer threads
# (src/data/assemble/sql_engine.py) -- pd.read_parquet() reads the whole
# directory of them in one call, no glob needed.
assembled_tile = (
    f"{DATA}/assembled/grid={grid_label}/shake={shake_label}/ix={ix}/iy={iy}"
)

In [ ]:
# --- load the panel tile + the geobox it was cut from ----------------------
parquet_tile = pd.read_parquet(assembled_tile)

# ix/iy tiling is always on the native 1km canonical grid (2048x2048 native
# cells per tile); a coarser grid_label just reprojects that same tile down
# with zoom_to(), which is what sql_engine._pixel_id_sql's ragged per-tile
# block aggregation mirrors on the SQL side.
native_geobox = canonical_ease_geobox()
tile_native = create_tile_geobox(native_geobox, DEFAULT_TILE_SIZE, ix, iy)
resolution_m = GRID_RESOLUTIONS_M[grid_label]
tile = tile_native if resolution_m == 1000.0 else tile_native.zoom_to(resolution=resolution_m)
dim_y, dim_x = geobox_spatial_dims(tile)  # ('y', 'x') -- EASE6933 is projected, not lat/lon

print("panel rows :", len(parquet_tile))
print("columns    :", list(parquet_tile.columns))
if "year" in parquet_tile.columns:
    yrs = np.sort(parquet_tile["year"].dropna().unique())
    print("years      :", yrs.min(), "..", yrs.max(), f"({len(yrs)} unique)")

In [ ]:
# --- pixel_id -> (y, x) for this tile ---------------------------------------
pixel_id_ds = make_pixel_ids(ix, iy, tile)
conversion_df = (
    pixel_id_ds["pixel_id"].to_dataframe().reset_index()[["pixel_id", dim_y, dim_x]]
)

# tile footprint in lon/lat, just for the figure title
transformer = pyproj.Transformer.from_crs(tile.crs, "EPSG:4326", always_xy=True)
(lon0, lon1), (lat0, lat1) = transformer.transform(
    [tile.boundingbox.left, tile.boundingbox.right],
    [tile.boundingbox.bottom, tile.boundingbox.top],
)

In [ ]:
# --- pick the columns to plot, classifying each for the right color encoding -
# A continuous colormap on a nominal ID column draws a fake gradient across
# arbitrary integer codes (this was happening to GID_*/biome_id/eco_id/
# lccs_class/flare_band, which are dtype uint/float, not object, so the old
# object-dtype check missed them entirely). Three buckets instead:
#   - continuous (physical magnitude)        -> one sequential hue
#   - low-cardinality nominal (<=20 levels)   -> discrete qualitative palette,
#                                                colorbar ticks show the real code
#   - high-cardinality nominal (admin units:
#     GID_1..GID_5 run into the thousands)    -> no legend could list that many
#     codes anyway, so each code's hue is hashed instead, just to keep
#     neighbouring polygons visually distinct
NOMINAL_HINTS = {
    "GID_0", "GID_1", "GID_2", "GID_3", "GID_4", "GID_5",
    "realm_id", "biome_id", "eco_id", "lccs_class", "flare_band",
}
HIGH_CARDINALITY_THRESHOLD = 20

id_cols = {"pixel_id", "year", "ix", "iy", dim_y, dim_x}
value_cols = [c for c in parquet_tile.columns if c not in id_cols]

panel = parquet_tile
if "year" in panel.columns:
    panel = panel[panel["year"] == year]
    if panel.empty:
        raise ValueError(f"no panel rows for year={year}")

# drop columns that are entirely missing for this tile/year
value_cols = [c for c in value_cols if panel[c].notna().any()]

encoded = panel[["pixel_id"]].copy()
continuous_cols, low_card_cols, high_card_cols = [], [], []
low_card_levels = {}  # col -> sorted real category codes, for colorbar tick labels

for c in value_cols:
    s = panel[c]
    is_nominal = (
        c in NOMINAL_HINTS
        or s.dtype == object
        or str(s.dtype).startswith("category")
        or s.dtype == bool
    )
    if not is_nominal:
        encoded[c] = s.astype("float32")
        continuous_cols.append(c)
        continue

    if s.dropna().nunique() > HIGH_CARDINALITY_THRESHOLD:
        encoded[c] = pd.to_numeric(s, errors="coerce").astype("float32")
        high_card_cols.append(c)
    else:
        levels = sorted(s.dropna().unique().tolist())
        code_to_index = {v: i for i, v in enumerate(levels)}
        encoded[c] = s.map(code_to_index).astype("float32")
        low_card_cols.append(c)
        low_card_levels[c] = levels

print(
    f"{len(continuous_cols)} continuous, {len(low_card_cols)} low-cardinality "
    f"categorical, {len(high_card_cols)} high-cardinality categorical (hashed)"
)

In [ ]:
# --- fold every column back onto the tile grid ------------------------------
grid = (
    conversion_df.merge(encoded, on="pixel_id", how="left")
    .set_index([dim_y, dim_x])[value_cols]
    .to_xarray()
)
grid = grid.odc.assign_crs(tile.crs)
grid

In [ ]:
# --- one imshow per source -------------------------------------------------
GOLDEN_RATIO_CONJUGATE = 0.6180339887498949  # spreads any integer code evenly around the hue wheel


def hash_to_rgba(values: np.ndarray) -> np.ndarray:
    """Deterministic code -> color hash for nominal columns with too many
    levels to legend (GID_2..GID_5-style polygon IDs). Adjacent integer codes
    land far apart on the hue wheel so neighbouring polygons stay visually
    distinct; the specific color carries no meaning and gets no colorbar."""
    values = np.asarray(values, dtype="float64")
    rgba = np.ones(values.shape + (4,), dtype="float32")
    valid = ~np.isnan(values)
    hues = (values[valid].astype("int64") * GOLDEN_RATIO_CONJUGATE) % 1.0
    hsv = np.stack([hues, np.full(hues.shape, 0.55), np.full(hues.shape, 0.92)], axis=-1)
    rgba[valid, :3] = mcolors.hsv_to_rgb(hsv)
    rgba[~valid] = (1.0, 1.0, 1.0, 0.0)  # transparent where masked/missing
    return rgba


all_cols = continuous_cols + low_card_cols + high_card_cols
ncols = 3
nrows = int(np.ceil(len(all_cols) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.4 * nrows))
axes = np.atleast_1d(axes).ravel()

for ax, col in zip(axes, all_cols):
    da = grid[col]
    if col in high_card_cols:
        rgba_da = xr.DataArray(
            hash_to_rgba(da.values),
            dims=(dim_y, dim_x, "channel"),
            coords={dim_y: da[dim_y], dim_x: da[dim_x]},
        )
        rgba_da.plot.imshow(ax=ax, rgb="channel", add_labels=False)
        ax.set_title(f"{col} ({int(da.to_series().nunique())} codes, hashed)", fontsize=9)
    elif col in low_card_cols:
        levels = low_card_levels[col]
        n = len(levels)
        cmap = plt.get_cmap("tab20", max(n, 1))
        im = da.plot.imshow(
            ax=ax, cmap=cmap, vmin=-0.5, vmax=n - 0.5, add_labels=False, add_colorbar=False
        )
        cb = fig.colorbar(im, ax=ax, ticks=range(n), fraction=0.046, pad=0.03)
        cb.ax.set_yticklabels([str(v) for v in levels], fontsize=6)
        ax.set_title(f"{col} (codes)", fontsize=9)
    else:
        da.plot.imshow(ax=ax, robust=True, cmap="cividis", add_labels=False)
        ax.set_title(col, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

for ax in axes[len(all_cols):]:
    ax.set_visible(False)

fig.suptitle(
    f"assembled {grid_label} panel  -  tile ix={ix} iy={iy}  -  year {year}\n"
    f"lon [{lon0:.2f}, {lon1:.2f}]  lat [{lat0:.2f}, {lat1:.2f}]",
    y=1.02,
)
fig.tight_layout()